In [71]:
%pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [72]:
%pip install "transformers[torch]"

Note: you may need to restart the kernel to use updated packages.


In [73]:
!python -m pip install torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [74]:
import sys
import types

# Mock torchaudio as a dummy module so transformers passes its check without loading C++ DLLs
sys.modules['torchaudio'] = types.ModuleType('torchaudio')

import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, TrainingArguments, Trainer

In [75]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [76]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [77]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [78]:
train_data.shape

(14732, 3)

In [79]:
val_data.shape

(818, 3)

In [80]:
# random sampling
train_data=train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [81]:
# Data pre-processing
import re

def clean_data(text):
    text=re.sub(r"\r\n", " ", text) # line
    text=re.sub(r"\s+", " ", text) # space
    text=re.sub(r"<.>*?"," ", text)# html tags <p> <h1>
    text=text.strip().lower()
    return text

In [82]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [83]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:  ile_other> claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [84]:
# tokenization
tokenizer=T5Tokenizer.from_pretrained("t5-small")


In [85]:
# Raw data tokenizer function for fine-tuning
def tokenize(data):
    # 1. Tokenize input sequence
    inputs = tokenizer(
        data["dialogue"], 
        padding="max_length", 
        max_length=512, 
        truncation=True
    )
    
    # 2. Tokenize target sequence (summaries/labels)
    # Use text_target for target sequences to handle task-specific tokens properly
    target = tokenizer(
        text_target=data["summary"], 
        padding="max_length", 
        max_length=128,  # Summaries are typically shorter than inputs
        truncation=True
    )

    # 3. Add target token IDs as labels
    inputs["labels"] = target["input_ids"]
    
    return inputs

In [86]:
# Apply the function row-by-row and convert to a list
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [87]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 699, 834, 9269, 3155, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [88]:
  len(train_dataset[0]["input_ids"])

512

In [89]:
type(train_dataset)

list

In [90]:
#Working witht the model
model= T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [91]:
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device:", device)
model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [92]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./result",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",  # or evaluation_strategy="epoch" depending on your transformers version
    save_strategy="epoch",

    warmup_steps=500
)

In [93]:
# defining training model
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)